### 2 LAB Романов Михаил ПМИ-3

In [1]:
import re
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#### 1 Задание

Используя BeautifulSoup, извлечь с разрешённой статической учебной страницы или предоставленного HTML-файла: https://books.toscrape.com/ - ссылка на сайт для scraping


In [ ]:
# Базовый URL учебного сайта
BASE_URL = "https://books.toscrape.com/"

# Словарь для конвертации строкового рейтинга в число
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def create_session_with_retries(
    retries: int = 3, backoff_factor: float = 0.5
) -> requests.Session:
    # Создаёт сессию с автоматическими повторными попытками при сбоях (Retry)
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(
        {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
    )
    return session


def scrape_books(target_count: int = 20):
    session = create_session_with_retries(retries=3, backoff_factor=0.5)
    books = []
    current_url = BASE_URL

    while current_url and len(books) < target_count:
        try:
            # Выполняем GET-запрос с таймаутом в 10 секунд
            response = session.get(current_url, timeout=10)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Ошибка при запросе к {current_url}: {e}")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        articles = soup.select("article.product_pod")

        for article in articles:
            if len(books) >= target_count:
                break

            # 1. Ссылка на карточку книги
            link_tag = article.select_one("h3 > a")
            rel_url = link_tag.get("href", "")
            full_url = urljoin(current_url, rel_url)

            # 2. Название книги
            title = link_tag.get("title") or link_tag.get_text()
            title = " ".join(title.split())

            # 3. Числовой показатель: цена 
            price_elem = article.select_one("p.price_color")
            price_text = price_elem.get_text() if price_elem else ""
            price_match = re.search(r"[\d.]+", price_text)
            price = float(price_match.group(0)) if price_match else None

            # 4. Числовой показатель рейтинг (от 1 до 5)
            rating_elem = article.select_one("p.star-rating")
            rating = None
            if rating_elem:
                classes = rating_elem.get("class", [])
                for cls in classes:
                    if cls in RATING_MAP:
                        rating = int(RATING_MAP[cls])
                        break

            # 5. Идентификатор (data-* атрибут или уникальный id из URL)
            data_id = (
                article.get("data-id")
                or article.get("data-item-id")
                or article.get("data-book-id")
            )
            if not data_id:
                # В структуре books.toscrape.com id закодирован в слаге url (напр. /slug_1000/index.html)
                match = re.search(r"_(\d+)/index\.html", full_url)
                data_id = match.group(1) if match else None

            # 6. Категория книги (извлекается из детальной страницы книги)
            category = None
            try:
                detail_resp = session.get(full_url, timeout=10)
                if detail_resp.status_code == 200:
                    detail_soup = BeautifulSoup(
                        detail_resp.text, "html.parser"
                    )
                    # Breadcrumbs: Home > Books > Category > Title
                    crumbs = detail_soup.select(
                        "ul.breadcrumb > li:nth-child(3) > a"
                    )
                    if crumbs:
                        category = crumbs[0].get_text(strip=True)
            except requests.exceptions.RequestException:
                pass

            books.append(
                {
                    "title": title,
                    "category": category,
                    "price": price,
                    "rating": rating,
                    "url": full_url,
                    "data_id": data_id,
                }
            )

        # Пагинация (переход на следующую страницу)
        next_btn = soup.select_one("li.next > a")
        if next_btn:
            current_url = urljoin(current_url, next_btn.get("href"))
        else:
            current_url = None

    return books


#Запуск сбора, валидация и постобработка
books_data = scrape_books(target_count=20)
df = pd.DataFrame(books_data)

# Очистка и удаление дубликатов
df.drop_duplicates(subset=["url"], inplace=True)
df.reset_index(drop=True, inplace=True)

# Сохранение в CSV
csv_filename = "books_data.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

# Вывод результатов
print(f"Всего строк: {len(df)}")
print("\nДоля пропусков по колонкам (NaN ratio):")
print(df.isna().mean().round(4))
print("\nПервые 5 записей:")
print(df.head())

Всего строк: 20

Доля пропусков по колонкам (NaN ratio):
title       0.0
category    0.0
price       0.0
rating      0.0
url         0.0
data_id     0.0
dtype: float64

Первые 5 записей:
                                   title            category  price  rating  \
0                   A Light in the Attic              Poetry  51.77       3   
1                     Tipping the Velvet  Historical Fiction  53.74       1   
2                             Soumission             Fiction  50.10       1   
3                          Sharp Objects             Mystery  47.82       4   
4  Sapiens: A Brief History of Humankind             History  54.23       5   

                                                 url data_id  
0  https://books.toscrape.com/catalogue/a-light-i...    1000  
1  https://books.toscrape.com/catalogue/tipping-t...     999  
2  https://books.toscrape.com/catalogue/soumissio...     998  
3  https://books.toscrape.com/catalogue/sharp-obj...     997  
4  https://books.toscra

#### Part B. No-code (Octoparse)

Рабочий процесс для той же страницы `https://books.toscrape.com/` сохранён в `2lab/nocode/workflow.json`.

1. **Платформа**: Octoparse, режим List (статическая страница).
2. **Повторяющаяся запись (Loop item)**: `article.product_pod` — каждая карточка книги в каталоге.
3. **Извлекаемые поля (6, требование ≥ 4)**:
   - `title` — `h3 > a` (атрибут `title`, затем текст);
   - `url` — `h3 > a` → `href` (приводится к абсолютному);
   - `price` — `p.price_color` → текст, число после чистки пробелов;
   - `rating` — `p.star-rating` → класс (`star-rating One..Five` → 1..5);
   - `data_id` — `article.product_pod` → id из слага URL `_NNN/index.html`;
   - `category` — хлебные крошки детальной страницы `ul.breadcrumb > li:nth-child(3) > a`.
4. **Пагинация**: `li.next > a` — «Loop Click until the element disappears» (на каждой странице каталога есть кнопка Next, всего 50 страниц).
5. **Надёжность**: timeout 10 000 мс, 3 повторные попытки, waits 1000 мс перед действием и 500 мс после перехода на страницу.
6. **Тестовый запуск**: выполнен, результат экспортирован в `2lab/nocode/nocode_result.csv` (UTF-8).
7. **Скриншоты**: workflow, запуск и таблица результата приложены ниже.


##### Скриншоты nocode части

**Workflow (схема задачи, loop item и поля):**

![Octoparse workflow](nocode/screenshot_workflow.png)

**Тестовый запуск:**

![Octoparse run](nocode/screenshot_run.png)

**Таблица результата (экспорт):**

![Octoparse result](nocode/screenshot_final.png)


In [ ]:
# Сравнение А программной реализации  и В nocode экспорта
df_b = pd.read_csv("nocode/nocode_result.csv")

# Часть 1 собирает id/rating кодом, поэтому приводим их к числовым типам и там
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce").astype("Int64")
df["data_id"] = pd.to_numeric(df["data_id"], errors="coerce").astype("Int64")

# Приведение типов и чистка строк в nocode выгрузке (в Octoparse текст выгружается как есть)
for col in ["title", "category", "url"]:
    df_b[col] = df_b[col].astype("string").str.strip()
df_b["price"] = pd.to_numeric(df_b["price"], errors="coerce")
df_b["rating"] = pd.to_numeric(df_b["rating"], errors="coerce").astype("Int64")
df_b["data_id"] = pd.to_numeric(df_b["data_id"], errors="coerce").astype("Int64")
df_b = df_b.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Сравнение Part A (BeautifulSoup) vs Part B (Octoparse)")
print(f"{'Показатель':<30} {'Part A':>10} {'Part B':>10}")
print("-" * 52)
print(f"{'Число записей':<30} {len(df):>10} {len(df_b):>10}")
print(f"{'Число полей':<30} {df.shape[1]:>10} {df_b.shape[1]:>10}")
print(f"{'Набор полей совпадает':<30} {str(set(df.columns) == set(df_b.columns)):>10}")
print(f"{'Пропусков (NaN)':<30} {int(df.isna().sum().sum()):>10} {int(df_b.isna().sum().sum()):>10}")
print(f"{'Дубликатов по url':<30} {int(df.duplicated(subset=['url']).sum()):>10} {int(df_b.duplicated(subset=['url']).sum()):>10}")

# Совпадение самих записей
a_keys, b_keys = set(df["url"]), set(df_b["url"])
print(f"\nОбщих записей по url: {len(a_keys & b_keys)}; только в A: {len(a_keys - b_keys)}; только в B: {len(b_keys - a_keys)}")

# Сходимость числовых полей по общим записям
merged = df.merge(df_b, on="url", suffixes=("_a", "_b"))
diff_price = int((merged["price_a"] != merged["price_b"]).sum())
diff_rating = int((merged["rating_a"] != merged["rating_b"]).sum())
diff_id = int((merged["data_id_a"] != merged["data_id_b"]).sum())
print(f"Расхождения price/rating/data_id: {diff_price} / {diff_rating} / {diff_id}")

print("\nПервые 5 записей Part B:")
print(df_b.head())


Сравнение Part A (BeautifulSoup) vs Part B (Octoparse)
Показатель                         Part A     Part B
----------------------------------------------------
Число записей                          20         20
Число полей                             6          6
Набор полей совпадает                True
Пропусков (NaN)                         0          0
Дубликатов по url                       0          0

Общих записей по url: 20; только в A: 0; только в B: 0
Расхождения price/rating/data_id: 0 / 0 / 0

Первые 5 записей Part B:
                                   title            category  price  rating  \
0                   A Light in the Attic              Poetry  51.77       3   
1                     Tipping the Velvet  Historical Fiction  53.74       1   
2                             Soumission             Fiction  50.10       1   
3                          Sharp Objects             Mystery  47.82       4   
4  Sapiens: A Brief History of Humankind             History  54

##### Вывод по сравнению

- Обе реализации собрали по **20 записей** (требование «не менее 20» выполнено) и используют одинаковый набор из **6 полей**: `title`, `category`, `price`, `rating`, `url`, `data_id`.
- Ключ сравнения бизнес-ключ `url` (id книги `_NNN` в слаге): записи сходятся, расхождений в `price`/`rating`/`data_id` нет.
- Часть A (BeautifulSoup) даёт полный контроль: `timeout=10 c`, стратегия повторов `Retry(total=3, backoff=0.5)` для 429/5xx, очистка пробелов, приведение типов, `drop_duplicates`, сводка пропусков.
- Часть B (no-code) получает тот же результат без кода: loop по `article.product_pod`, пагинация по `li.next > a`, 6 полей, экспорт CSV; приведение типов и удаление дубликатов выполняется уже после экспорта (в скрипте сравнения выше).
- Пагинация в обеих частях реализована одинаково (кнопка Next), поэтому набор и порядок записей совпадают.
